In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)


# for visualization
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import re
import string
from scipy import stats

# Data preprocessing and feature engineering
from sklearn.preprocessing import StandardScaler, LabelEncoder, MinMaxScaler
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

# Model imports - 7 different models
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

# Evaluation and validation
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score
import sklearn.metrics as metrics

# Handling imbalanced data
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler

# Serialization
import pickle

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Loading The Dataset

In [3]:
sample_df = pd.read_csv("Sample.csv")
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

# Examine The Dataset & Exploratory Data Analysis

In [4]:
print("="*40, "Sample Data Shape", "="*40)
print("Shape: ", sample_df.shape)

print("="*40, "Sample Data Head", "="*40)
print(sample_df.head())

print("="*40, "Sample Data Info", "="*40)
print(sample_df.info())

======================================== Sample Data Shape ========================================
Shape:  (102000, 2)
======================================== Sample Data Head ========================================
   ID  label
0   1      0
1   2      0
2   3      0
3   4      0
4   5      0
======================================== Sample Data Info ========================================
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 102000 entries, 0 to 101999
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype
---  ------  --------------   -----
 0   ID      102000 non-null  int64
 1   label   102000 non-null  int64
dtypes: int64(2)
memory usage: 1.6 MB
None


In [5]:
print("="*40, "Train Data Shape", "="*40)
print("Shape: ", train_df.shape)

print("="*40, "Train Data Head", "="*40)
print(train_df.head())

print("="*40, "Train Data Info", "="*40)
print(train_df.info())

======================================== Train Data Shape ========================================
Shape:  (198000, 15)
======================================== Train Data Head ========================================
                       created_date  post_id  emoticon_1  emoticon_2  \
0  2024-01-18 08:43:57.397508+00:00       73           0           0   
1  2024-03-24 21:43:11.490017+00:00       39           0           0   
2  2024-04-24 20:32:17.014931+00:00       31           0           1   
3  2023-05-28 22:00:14.214527+00:00       39           0           0   
4  2023-09-09 23:12:05.689498+00:00       39           0           0   

   emoticon_3  upvote  downvote  if_1  if_2 race religion gender  disability  \
0           0       0         1     0    10  NaN      NaN    NaN       False   
1           0       6         0     0     4  NaN      NaN    NaN       False   
2           1       0         0     0    10  NaN      NaN    NaN       False   
3           0       5        

In [6]:
print("="*40, "Test Data Shape", "="*40)
print("Shape: ", test_df.shape)

print("="*40, "Test Data Head", "="*40)
print(test_df.head())

print("="*40, "Test Data Info", "="*40)
print(test_df.info())

======================================== Test Data Shape ========================================
Shape:  (102000, 14)
======================================== Test Data Head ========================================
                       created_date  post_id  emoticon_1  emoticon_2  \
0  2024-02-08 13:13:27.998156+00:00       72           2           0   
1  2024-03-01 23:33:25.547123+00:00      123           0           0   
2  2024-02-09 21:52:48.426303+00:00      120           0           0   
3  2024-02-17 03:43:02.980294+00:00      123           0           0   
4  2024-04-24 02:27:57.145155+00:00      123           0           0   

   emoticon_3  upvote  downvote  if_1  if_2 race religion gender  disability  \
0           0       4         1     0    10  NaN      NaN    NaN       False   
1           0       0         0     0    10  NaN      NaN    NaN       False   
2           0       3         0     0     4  NaN      NaN    NaN       False   
3           0       0         0

In [7]:
# info of sample dataframe
info_df = pd.DataFrame({
    'Column':sample_df.columns,
    'Data Type':sample_df.dtypes,
    'Non null count': sample_df.count(),
    'Null Count': sample_df.isnull().sum(),
    'Null %': (sample_df.isnull().sum()/ len(sample_df) * 100).round(2),
    'Unique Values': sample_df.nunique()
})

print(info_df.to_string(index = False))

Column Data Type  Non null count  Null Count  Null %  Unique Values
    ID     int64          102000           0     0.0         102000
 label     int64          102000           0     0.0              1


In [8]:
# info of train dataframe
info_df = pd.DataFrame({
    'Column':train_df.columns,
    'Data Type':train_df.dtypes,
    'Non null count': train_df.count(),
    'Null Count': train_df.isnull().sum(),
    'Null %': (train_df.isnull().sum()/ len(train_df) * 100).round(2),
    'Unique Values': train_df.nunique()
})

print(info_df.to_string(index = False))

      Column Data Type  Non null count  Null Count  Null %  Unique Values
created_date    object          198000           0    0.00         197996
     post_id     int64          198000           0    0.00             52
  emoticon_1     int64          198000           0    0.00             36
  emoticon_2     int64          198000           0    0.00             10
  emoticon_3     int64          198000           0    0.00             16
      upvote     int64          198000           0    0.00            122
    downvote     int64          198000           0    0.00             62
        if_1     int64          198000           0    0.00             57
        if_2     int64          198000           0    0.00             81
        race    object           52577      145423   73.45              6
    religion    object           52577      145423   73.45              8
      gender    object           52577      145423   73.45              5
  disability      bool          198000

In [9]:
# info of test dataframe
info_df = pd.DataFrame({
    'Column':test_df.columns,
    'Data Type':test_df.dtypes,
    'Non null count': test_df.count(),
    'Null Count': test_df.isnull().sum(),
    'Null %': (test_df.isnull().sum()/ len(test_df) * 100).round(2),
    'Unique Values': test_df.nunique()
})

print(info_df.to_string(index = False))

      Column Data Type  Non null count  Null Count  Null %  Unique Values
created_date    object          102000           0    0.00         101997
     post_id     int64          102000           0    0.00             45
  emoticon_1     int64          102000           0    0.00             29
  emoticon_2     int64          102000           0    0.00              8
  emoticon_3     int64          102000           0    0.00             16
      upvote     int64          102000           0    0.00             97
    downvote     int64          102000           0    0.00             47
        if_1     int64          102000           0    0.00             37
        if_2     int64          102000           0    0.00             70
        race    object           26731       75269   73.79              6
    religion    object           26731       75269   73.79              8
      gender    object           26731       75269   73.79              5
  disability      bool          102000

In [10]:
# missing values
print("                   MISSING VALUES ANALYSIS")
print("="*60)

missing = train_df.isnull().sum()
missing_percent = (missing / len(train_df)) * 100

missing_df = pd.DataFrame({
    'Collumn': train_df.columns,
    'Missing Count': missing.values,
    'Missing Percentage': missing_percent.values
})

missing_df


if missing.sum() > 0:
    print("There are missing values in the dataset.")
else:
    print("There are no missing values in the dataset.")


                   MISSING VALUES ANALYSIS
There are missing values in the dataset.


In [11]:
# Target variable exploration

print("           Target Variable Analysis: label")
print("="*60)

print(f"\nTarget Variable: 'label'")
print(f"Unique Classes: {train_df['label'].nunique()}")
print(f"Classes: {sorted(train_df['label'].unique())}")

# class distribution
class_counts = train_df['label'].value_counts().sort_index()
# class_counts

for cls, count in class_counts.items():
    pct = 100 * (count / len(train_df))
    bar = '█' * int(pct / 5)   # reduce the count 5% = 1 bar
    print(f"  Class {cls}: {count:6d} ({pct:6.2f}%) {bar}")

# Check for class imbalance
imbalance_ratio = class_counts.max() / class_counts.min()
print(f"\n Class Imbalance Ratio: {imbalance_ratio:.2f}")
if imbalance_ratio > 1.5:
    print(f" Classes are imbalanced ")
else:
    print(f" Classes are relatively balanced")

           Target Variable Analysis: label

Target Variable: 'label'
Unique Classes: 4
Classes: [0, 1, 2, 3]
  Class 0: 114173 ( 57.66%) ███████████
  Class 1:  15918 (  8.04%) █
  Class 2:  62440 ( 31.54%) ██████
  Class 3:   5469 (  2.76%) 

 Class Imbalance Ratio: 20.88
 Classes are imbalanced 


In [12]:

print("TEXT FEATURE ANALYSIS (comment column)")
print("="*60)

text_data = train_df['comment'].astype(str)

print(f"\nTotal Comments: {len(text_data)}")
print(f"\nText Length Statistics:")
print(f"  Min:     {text_data.str.len().min():6.0f} characters")
print(f"  Max:     {text_data.str.len().max():6.0f} characters")
print(f"  Mean:    {text_data.str.len().mean():6.0f} characters")
print(f"  Median:  {text_data.str.len().median():6.0f} characters")
print(f"  Std Dev: {text_data.str.len().std():6.0f} characters")

word_counts = text_data.str.split().str.len()
print(f"\nWord Count Statistics:")
print(f"  Min:     {word_counts.min():6.0f} words")
print(f"  Max:     {word_counts.max():6.0f} words")
print(f"  Mean:    {word_counts.mean():6.0f} words")
print(f"  Median:  {word_counts.median():6.0f} words")
print(f"  Std Dev: {word_counts.std():6.0f} words")

TEXT FEATURE ANALYSIS (comment column)

Total Comments: 198000

Text Length Statistics:
  Min:          1 characters
  Max:       1892 characters
  Mean:       303 characters
  Median:     211 characters
  Std Dev:    266 characters

Word Count Statistics:
  Min:          1 words
  Max:        315 words
  Mean:        52 words
  Median:      37 words
  Std Dev:     46 words


In [13]:
print("NUMERICAL FEATURES ANALYSIS")
print("="*60)

numerical_cols = train_df.select_dtypes(include=[np.number]).columns.tolist()
if 'label' in numerical_cols:
    numerical_cols.remove('label')  # Exclude target

print(f"\nNumerical Columns ({len(numerical_cols)}):")
for col in numerical_cols:
    print(f"  - {col}")

if len(numerical_cols) > 0:
    print(f"\nNumerical Features Statistics:")
    stats_df = train_df[numerical_cols].describe()
    print(stats_df.to_string())
    
    print(f"\nSkewness Analysis (measures asymmetry):")
    for col in numerical_cols:
        skewness = train_df[col].skew()
        print(f"  {col:20s}: {skewness:7.4f}", end="")
        if abs(skewness) > 1:
            print(" (Highly Skewed)")
        elif abs(skewness) > 0.5:
            print(" (Moderately Skewed)")
        else:
            print(" (Approximately Symmetric)")

print("SECTION 3.6: CATEGORICAL FEATURES ANALYSIS")
print("-"*80)

categorical_cols = train_df.select_dtypes(include=['object']).columns.tolist()
if 'comment' in categorical_cols:
    categorical_cols.remove('comment')  # Exclude text column
if 'created_date' in categorical_cols:
    categorical_cols.remove('created_date')  # Exclude date column

print(f"\nCategorical Columns ({len(categorical_cols)}):")
for col in categorical_cols:
    unique_count = train_df[col].nunique()
    print(f"  - {col:20s}: {unique_count} unique values")

for col in categorical_cols:
    print(f"\n{col} Distribution:")
    dist = train_df[col].value_counts()
    for val, count in dist.head(10).items():
        pct = 100 * count / len(train_df)
        print(f"  {str(val):20s}: {count:6d} ({pct:5.1f}%)")
    if len(dist) > 10:
        print(f"  ... and {len(dist) - 10} more values")

NUMERICAL FEATURES ANALYSIS

Numerical Columns (8):
  - post_id
  - emoticon_1
  - emoticon_2
  - emoticon_3
  - upvote
  - downvote
  - if_1
  - if_2

Numerical Features Statistics:
             post_id     emoticon_1     emoticon_2     emoticon_3         upvote       downvote           if_1           if_2
count  198000.000000  198000.000000  198000.000000  198000.000000  198000.000000  198000.000000  198000.000000  198000.000000
mean       68.447429       0.279768       0.048338       0.121071       2.607975       0.666394       1.906152       7.956212
std        27.948390       1.023234       0.258477       0.481013       5.054763       2.044335      25.635752      14.839464
min        20.000000       0.000000       0.000000       0.000000       0.000000       0.000000       0.000000       3.000000
25%        39.000000       0.000000       0.000000       0.000000       0.000000       0.000000       0.000000       4.000000
50%        72.000000       0.000000       0.000000       0.00

In [14]:
print("CATEGORICAL FEATURES ANALYSIS")
print("-"*80)

categorical_cols = train_df.select_dtypes(include=['object']).columns.tolist()
if 'comment' in categorical_cols:
    categorical_cols.remove('comment')  # Exclude text column
if 'created_date' in categorical_cols:
    categorical_cols.remove('created_date')  # Exclude date column

print(f"\nCategorical Columns ({len(categorical_cols)}):")
for col in categorical_cols:
    unique_count = train_df[col].nunique()
    print(f"  - {col:20s}: {unique_count} unique values")

for col in categorical_cols:
    print(f"\n{col} Distribution:")
    dist = train_df[col].value_counts()
    for val, count in dist.head(10).items():
        pct = 100 * count / len(train_df)
        print(f"  {str(val):20s}: {count:6d} ({pct:5.1f}%)")
    if len(dist) > 10:
        print(f"  ... and {len(dist) - 10} more values")

CATEGORICAL FEATURES ANALYSIS
--------------------------------------------------------------------------------

Categorical Columns (3):
  - race                : 6 unique values
  - religion            : 8 unique values
  - gender              : 5 unique values

race Distribution:
  none                :  39682 ( 20.0%)
  white               :   5486 (  2.8%)
  black               :   3869 (  2.0%)
  other               :   1654 (  0.8%)
  asian               :   1263 (  0.6%)
  latino              :    623 (  0.3%)

religion Distribution:
  none                :  38249 ( 19.3%)
  christian           :   7191 (  3.6%)
  muslim              :   4930 (  2.5%)
  jewish              :   1244 (  0.6%)
  other               :    547 (  0.3%)
  atheist             :    219 (  0.1%)
  buddhist            :    100 (  0.1%)
  hindu               :     97 (  0.0%)

gender Distribution:
  none                :  36161 ( 18.3%)
  female              :   8037 (  4.1%)
  male                :   7549 

# data preprocessing and feature engineering

In [15]:
# ============================================================================
# SECTION 4.1: Text Cleaning Function
# ============================================================================
print("DATA PREPROCESSING AND FEATURE ENGINEERING")
print("="*80)

def clean_text(text):
    """
    Clean and preprocess text for feature extraction.
    
    Steps:
    1. Convert to lowercase
    2. Remove URLs (http, https, www)
    3. Remove email addresses
    4. Remove special characters (keep only alphanumeric and spaces)
    5. Remove extra whitespace
    
    """
    # Handle non-string values
    if not isinstance(text, str):
        return ""
    
    # Convert to lowercase
    text = text.lower()
    
    # Remove URLs (http://, https://, www.)
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    
    # Remove email addresses
        # S = any non white space. (any char)
    text = re.sub(r'\S+@\S+', '', text)
    
    # Remove special characters (keep only alphanumeric and spaces)
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# Test the cleaning function
print("\n--- Text Cleaning Function Test ---")
test_texts = [
    "This is GREAT! Check https://example.com",
    "Contact me at user@email.com for more",
    "Amazing product with 100% quality!!!"
]

for text in test_texts:
    cleaned = clean_text(text)
    print(f"Original: {text}")
    print(f"Cleaned:  {cleaned}\n")

DATA PREPROCESSING AND FEATURE ENGINEERING

--- Text Cleaning Function Test ---
Original: This is GREAT! Check https://example.com
Cleaned:  this is great check

Original: Contact me at user@email.com for more
Cleaned:  contact me at for more

Original: Amazing product with 100% quality!!!
Cleaned:  amazing product with 100 quality



In [16]:
print("CLEANING  TEXT IN TRAINING DATA")
train_df['cleaned_text'] = train_df['comment'].apply(clean_text)
test_df['cleaned_text'] = test_df['comment'].apply(clean_text)

print(f"Training cleaned text: {len(train_df)} length")
print(f"Testing cleaned text: {len(train_df)} length")

print("Sampleed Cleaned Texts: ")
for i in range(min(3, len(train_df))):
    # integer-location based indexing
    print(f"{i+1}. {train_df['cleaned_text'].iloc[i][:100]}...")

CLEANING  TEXT IN TRAINING DATA
Training cleaned text: 198000 length
Testing cleaned text: 198000 length
Sampleed Cleaned Texts: 
1. she might be a bright spot for a party keou on oahu dominated by greedy criminals or ethically chall...
2. under alaska law a nontribal member is not bound to tribal court living in a particular community do...
3. in the future please spare me your strawman drivel and if you cant manage an intelligent discussion ...


In [17]:
# Initializes vectorizers and extractors

tfidf_vectorizer = TfidfVectorizer(
    
    # keep top 500 featurs
    max_features = 500,

    # use unigram and biagram
    ngram_range = (1,2),

    # remove command english stop words
    stop_words = 'english',

    # words must appear in atleas 2 docs
    min_df = 2,

    # word must not appear in more than 95% of doc
    max_df = 0.95    
)


# Count Vectorizer (Bag of Words)
# Counts occurrences of words
count_vectorizer = CountVectorizer(
    
    # Keep top 300 features
    max_features=300,              
    
    # Unigrams and bigrams
    ngram_range=(1, 2),            
    
    stop_words='english',
    min_df=2,
    max_df=0.95
)

print("  TFIDF Vectorizer initialized (500 features)")
print("  Count Vectorizer initialized (300 features)")

  TFIDF Vectorizer initialized (500 features)
  Count Vectorizer initialized (300 features)


In [18]:
print("EXTRACT FEATURES")

# Extract TFIDF features (from training)
print("  Extracting TFIDF features from training data...")
tfidf_matrix_train = tfidf_vectorizer.fit_transform(train_df['cleaned_text'])

# Convert sparse matrix to dense array
tfidf_train = tfidf_matrix_train.toarray()  

print(f"    TFIDF shape: {tfidf_train.shape}")

EXTRACT FEATURES
  Extracting TFIDF features from training data...
    TFIDF shape: (198000, 500)


In [19]:
# Extract TF-IDF features (from test)
print("  Extracting TFIDF features from test data...")
tfidf_matrix_test = tfidf_vectorizer.transform(test_df['cleaned_text'])
tfidf_test = tfidf_matrix_test.toarray()
print(f"    Test TF-IDF shape: {tfidf_test.shape}")

  Extracting TFIDF features from test data...
    Test TF-IDF shape: (102000, 500)


In [20]:
# Extract Count features (from training)
print("  Extracting Count features from training data...")
count_matrix_train = count_vectorizer.fit_transform(train_df['cleaned_text'])
count_train = count_matrix_train.toarray()
print(f"    Count shape: {count_train.shape}")

  Extracting Count features from training data...
    Count shape: (198000, 300)


In [21]:
# Extract Count features (test)
print("  Extracting Count features from test data...")
count_matrix_test = count_vectorizer.transform(test_df['cleaned_text'])
count_test = count_matrix_test.toarray()
print(f"    Test Count shape: {count_test.shape}")

  Extracting Count features from test data...
    Test Count shape: (102000, 300)


In [22]:
print("Extracting Statistical Text Features")

def extract_stat_features(df):
    """
    Extract statistical features from text.
    
    Features:
    1. text_length: Total characters in text
    2. word_count: Number of words
    3. uppercase_count: Number of uppercase letters
    4. digit_count: Number of digits
    5. punctuation_count: Number of punctuation marks
    6. avg_word_length: Average length of words
    """
    
    return pd.DataFrame({
        'text_length': df['cleaned_text'].str.len(),
        'word_count': df['cleaned_text'].str.split().str.len(),
        'uppercase_count': df['cleaned_text'].str.count('[A-Z]'),
        'digit_count': df['cleaned_text'].str.count('\d'),
        'punctuation_count': df['cleaned_text'].str.count('[' + re.escape(string.punctuation) + ']'),
        'avg_word_length': df['cleaned_text'].str.split().apply(
            lambda x: np.mean([len(w) for w in x]) if len(x) > 0 else 0
        )
    }).fillna(0)

stat_features_train = extract_stat_features(train_df)
stat_features_test = extract_stat_features(test_df)

print(f"  Statistical features extracted: {stat_features_train.shape[1]} features")
print(f"  Feature Statistics (training data):")
print(stat_features_train.describe().to_string())

Extracting Statistical Text Features
  Statistical features extracted: 6 features
  Feature Statistics (training data):
         text_length     word_count  uppercase_count    digit_count  punctuation_count  avg_word_length
count  198000.000000  198000.000000         198000.0  198000.000000           198000.0    198000.000000
mean      289.524571      52.051308              0.0       1.024955                0.0         4.588915
std       256.358318      45.500027              0.0       3.302133                0.0         1.348828
min         0.000000       0.000000              0.0       0.000000                0.0         0.000000
25%        96.000000      18.000000              0.0       0.000000                0.0         4.218750
50%       201.000000      37.000000              0.0       0.000000                0.0         4.533333
75%       401.000000      72.000000              0.0       0.000000                0.0         4.868421
max      1807.000000     309.000000             

In [23]:
print("Extracting Numerical Features")

# Define numerical feature columns
numerical_feature_cols = [
    'emoticon_1', 'emoticon_2', 'emoticon_3',  
    'upvote', 'downvote',                       
    'if_1', 'if_2'                              
]

print(f"  Numerical columns: {numerical_feature_cols}")

# Extract numerical features
numerical_train = train_df[numerical_feature_cols].fillna(0)
numerical_test = test_df[numerical_feature_cols].fillna(0)

print(f"  Numerical features extracted: {numerical_train.shape[1]} features")
print(f"  Numerical Features Statistics:")

print(numerical_train.describe().to_string())

Extracting Numerical Features
  Numerical columns: ['emoticon_1', 'emoticon_2', 'emoticon_3', 'upvote', 'downvote', 'if_1', 'if_2']
  Numerical features extracted: 7 features
  Numerical Features Statistics:
          emoticon_1     emoticon_2     emoticon_3         upvote       downvote           if_1           if_2
count  198000.000000  198000.000000  198000.000000  198000.000000  198000.000000  198000.000000  198000.000000
mean        0.279768       0.048338       0.121071       2.607975       0.666394       1.906152       7.956212
std         1.023234       0.258477       0.481013       5.054763       2.044335      25.635752      14.839464
min         0.000000       0.000000       0.000000       0.000000       0.000000       0.000000       3.000000
25%         0.000000       0.000000       0.000000       0.000000       0.000000       0.000000       4.000000
50%         0.000000       0.000000       0.000000       1.000000       0.000000       0.000000       6.000000
75%         0.0

In [24]:
print(" Extracting Categorical Features (One-Hot Encoding)")

# Define categorical feature columns
categorical_feature_cols = ['race', 'religion', 'gender', 'disability']

print(f"  Categorical columns: {categorical_feature_cols}")

# One-hot encode categorical features
print(f"  One-hot encoding categorical features...")
categorical_train_encoded = pd.get_dummies(train_df[categorical_feature_cols], drop_first=True)
categorical_test_encoded = pd.get_dummies(test_df[categorical_feature_cols], drop_first=True)

# Ensure test set has same columns as training set
# Add missing columns with 0 values
for col in categorical_train_encoded.columns:
    if col not in categorical_test_encoded.columns:
        categorical_test_encoded[col] = 0

# Remove extra columns in test set
categorical_test_encoded = categorical_test_encoded[categorical_train_encoded.columns]

print(f"  Categorical features extracted: {categorical_train_encoded.shape[1]} features")
print(f"  Categorical Features:")
print(categorical_train_encoded.columns.tolist())

 Extracting Categorical Features (One-Hot Encoding)
  Categorical columns: ['race', 'religion', 'gender', 'disability']
  One-hot encoding categorical features...
  Categorical features extracted: 17 features
  Categorical Features:
['disability', 'race_black', 'race_latino', 'race_none', 'race_other', 'race_white', 'religion_buddhist', 'religion_christian', 'religion_hindu', 'religion_jewish', 'religion_muslim', 'religion_none', 'religion_other', 'gender_male', 'gender_none', 'gender_other', 'gender_transgender']


In [25]:
print("Combining All Features")

print(f"\n  Feature Summary:")
print(f"    TFIDF features:       {tfidf_train.shape[1]}")
print(f"    Count features:        {count_train.shape[1]}")
print(f"    Statistical features:  {stat_features_train.shape[1]}")
print(f"    Numerical features:    {numerical_train.shape[1]}")
print(f"    Categorical features:  {categorical_train_encoded.shape[1]}")

total_features = (tfidf_train.shape[1] + count_train.shape[1] + stat_features_train.shape[1] +
                 numerical_train.shape[1] + categorical_train_encoded.shape[1])

print(f"    " + "="*60)
print(f"    TOTAL FEATURES:         {total_features}")

# Combine all features into single arrays
X_train_combined = np.hstack([
    tfidf_train,
    count_train,
    stat_features_train.values,
    numerical_train.values,
    categorical_train_encoded.values
])

X_test_combined = np.hstack([
    tfidf_test,
    count_test,
    stat_features_test.values,
    numerical_test.values,
    categorical_test_encoded.values
])

print(f"\n  Combined feature matrix shapes:")
print(f"    Training: {X_train_combined.shape}")
print(f"    Test:     {X_test_combined.shape}")



Combining All Features

  Feature Summary:
    TFIDF features:       500
    Count features:        300
    Statistical features:  6
    Numerical features:    7
    Categorical features:  17
    TOTAL FEATURES:         830

  Combined feature matrix shapes:
    Training: (198000, 830)
    Test:     (102000, 830)


In [ ]:
print("Scaling Features (by StandardScaler)")

# StandardScaler: Transforms features to have mean=0, std=1
scaler = StandardScaler()

print(f"  Fitting scaler on training data...")
X_train = scaler.fit_transform(X_train_combined)

print(f"  Transforming test data...")
X_test = scaler.transform(X_test_combined)

print(f"    Features scaled successfully")
print(f"    Training features shape: {X_train.shape}")
print(f"    Test features shape:     {X_test.shape}")
print(f"   \n FEATURE ENGINEERING COMPLETE")
print(f"  Total features per sample: {X_train.shape[1]}")

Scaling Features (by StandardScaler)
  Fitting scaler on training data...


MemoryError: Unable to allocate 1.22 GiB for an array with shape (198000, 830) and data type float64

: 

In [ ]:
print("TARGET VARIABLE PREPARATION")
print("="*80)

# Extract target variable (label)
y_train = train_df['label'].values

print(f"\nTarget Variable: 'label'")
print(f"Data Type: {train_df['label'].dtype}")
print(f"Total Samples: {len(y_train)}")

print(f"\nClass Distribution:")
for cls in sorted(np.unique(y_train)):
    count = (y_train == cls).sum()
    pct = 100 * count / len(y_train)
    bar = '█' * int(pct / 5)
    print(f"  Class {cls}: {count:6d} ({pct:6.2f}%) {bar}")

print(f"\n Target variable ready for training")

In [ ]:
print("TRAIN-VALIDATION SPLIT (with Stratification)")
print("="*80)

print("\nPerforming 80-20 stratified split...")
print("  - 80% for training model")
print("  - 20% for validation")
print("  - Stratification maintains class distribution")

X_train_split, X_val, y_train_split, y_val = train_test_split(
    X_train, y_train,
    test_size=0.2,                  
    random_state=42,                
    stratify=y_train                
)

print(f"\nSplit Complete:")
print(f"  Total samples:      {len(X_train)}")
print(f"  Training samples:   {X_train_split.shape[0]} ({100*X_train_split.shape[0]/len(X_train):.1f}%)")
print(f"  Validation samples: {X_val.shape[0]} ({100*X_val.shape[0]/len(X_train):.1f}%)")
print(f"  Features per sample: {X_train_split.shape[1]}")
print(f"  Class distribution preserved in both sets")

# Logistic Regression

In [ ]:
print("MODEL 1:  LOGISTIC REGRESSION")
print("="*80)

print("\nModel: Linear classifier for binary/multiclass classification")
print("Use Case: Fast baseline model, interpretable")
print("Hyperparameters:")
print("  max_iter=1000 - Maximum iterations for convergence")
print("  random_state=42 - Reproducibility")

print("\nTraining...")
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_split, y_train_split)

print("Evaluating on validation set...")
y_pred_lr = lr.predict(X_val)
lr_accuracy = accuracy_score(y_val, y_pred_lr)
lr_precision = precision_score(y_val, y_pred_lr, average='weighted', zero_division=0)
lr_recall = recall_score(y_val, y_pred_lr, average='weighted', zero_division=0)
lr_f1 = f1_score(y_val, y_pred_lr, average='weighted', zero_division=0)

print(f"\nValidation Results:")
print(f"  Accuracy:  {lr_accuracy:.4f}")
print(f"  Precision: {lr_precision:.4f}")
print(f"  Recall:    {lr_recall:.4f}")
print(f"  F1-Score:  {lr_f1:.4f}")
print(f"\n✓ Logistic Regression trained successfully")

# Randome Forest

In [ ]:
print("MODEL 2: RANDOM FOREST")
print("="*80)

print("\nModel: Ensemble of decision trees")
print("Use Case: Non-linear patterns, feature importance")
print("Hyperparameters:")
print("  n_estimators=100 - Number of trees")
print("  max_depth=15 - Maximum tree depth")
print("  min_samples_split=5 - Minimum samples to split")
print("  min_samples_leaf=2 - Minimum samples in leaf")

print("\nTraining...")
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1  # Use all processors
)
rf.fit(X_train_split, y_train_split)

print("Evaluating on validation set...")
y_pred_rf = rf.predict(X_val)
rf_accuracy = accuracy_score(y_val, y_pred_rf)
rf_precision = precision_score(y_val, y_pred_rf, average='weighted', zero_division=0)
rf_recall = recall_score(y_val, y_pred_rf, average='weighted', zero_division=0)
rf_f1 = f1_score(y_val, y_pred_rf, average='weighted', zero_division=0)

print(f"\nValidation Results:")
print(f"  Accuracy:  {rf_accuracy:.4f}")
print(f"  Precision: {rf_precision:.4f}")
print(f"  Recall:    {rf_recall:.4f}")
print(f"  F1-Score:  {rf_f1:.4f}")
print(f"\nRandom Forest trained successfully")

# Support Vector Machine

In [ ]:
print("MODEL 3: SUPPORT VECTOR MACHINE (SVM) - MANDATORY")
print("="*80)

print("\nModel: Non-linear classifier with RBF kernel")
print("Status: MANDATORY 3rd model per MLP requirements")
print("Use Case: Non-linear decision boundaries")
print("Hyperparameters:")
print("  kernel='rbf' - Radial Basis Function (non-linear)")
print("  C=1.0 - Regularization parameter")
print("  gamma='scale' - Kernel coefficient")
print("  probability=True - Enable probability estimates")

print("\nTraining...")
svm = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42, probability=True)
svm.fit(X_train_split, y_train_split)

print("Evaluating on validation set...")
y_pred_svm = svm.predict(X_val)
svm_accuracy = accuracy_score(y_val, y_pred_svm)
svm_precision = precision_score(y_val, y_pred_svm, average='weighted', zero_division=0)
svm_recall = recall_score(y_val, y_pred_svm, average='weighted', zero_division=0)
svm_f1 = f1_score(y_val, y_pred_svm, average='weighted', zero_division=0)

print(f"\nValidation Results:")
print(f"  Accuracy:  {svm_accuracy:.4f}")
print(f"  Precision: {svm_precision:.4f}")
print(f"  Recall:    {svm_recall:.4f}")
print(f"  F1-Score:  {svm_f1:.4f}")
print(f"\n✓ SVM trained successfully (MANDATORY requirement met )")

# Gradient Boosting

In [ ]:
print("MODEL 4: GRADIENT BOOSTING")
print("="*80)

print("\nModel: Sequential ensemble of trees")
print("Use Case: Strong sequential learning, high performance")
print("Hyperparameters:")
print("  n_estimators=100 - Number of boosting stages")
print("  learning_rate=0.1 - Shrinkage parameter")
print("  max_depth=8 - Maximum tree depth")
print("  min_samples_split=5 - Minimum samples to split")
print("  min_samples_leaf=2 - Minimum samples in leaf")

print("\nTraining...")
gb = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=8,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42
)
gb.fit(X_train_split, y_train_split)

print("Evaluating on validation set...")
y_pred_gb = gb.predict(X_val)
gb_accuracy = accuracy_score(y_val, y_pred_gb)
gb_precision = precision_score(y_val, y_pred_gb, average='weighted', zero_division=0)
gb_recall = recall_score(y_val, y_pred_gb, average='weighted', zero_division=0)
gb_f1 = f1_score(y_val, y_pred_gb, average='weighted', zero_division=0)

print(f"\nValidation Results:")
print(f"  Accuracy:  {gb_accuracy:.4f}")
print(f"  Precision: {gb_precision:.4f}")
print(f"  Recall:    {gb_recall:.4f}")
print(f"  F1-Score:  {gb_f1:.4f}")
print(f"\nGradient Boosting trained successfully")

# XGBoost

In [ ]:
print("MODEL 5: XGBOOST")
print("="*80)

print("\nModel: Optimized gradient boosting framework")
print("Use Case: Fast training, excellent performance")
print("Hyperparameters:")
print("  n_estimators=100 - Number of boosting rounds")
print("  learning_rate=0.1 - Shrinkage parameter")
print("  max_depth=7 - Maximum tree depth")
print("  subsample=0.8 - Subsample ratio of training instances")
print("  colsample_bytree=0.8 - Subsample ratio of features")

print("\nTraining...")
xgb = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=7,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='mlogloss',
    verbosity=0
)
xgb.fit(X_train_split, y_train_split)

print("Evaluating on validation set...")
y_pred_xgb = xgb.predict(X_val)
xgb_accuracy = accuracy_score(y_val, y_pred_xgb)
xgb_precision = precision_score(y_val, y_pred_xgb, average='weighted', zero_division=0)
xgb_recall = recall_score(y_val, y_pred_xgb, average='weighted', zero_division=0)
xgb_f1 = f1_score(y_val, y_pred_xgb, average='weighted', zero_division=0)

print(f"\nValidation Results:")
print(f"  Accuracy:  {xgb_accuracy:.4f}")
print(f"  Precision: {xgb_precision:.4f}")
print(f"  Recall:    {xgb_recall:.4f}")
print(f"  F1-Score:  {xgb_f1:.4f}")
print(f"\n XGBoost trained successfully")

# LightBGM

In [ ]:
print("MODEL 6: LIGHTGBM")
print("="*80)

print("\nModel: Fast gradient boosting framework")
print("Use Case: Large datasets, memory efficient")
print("Hyperparameters:")
print("  n_estimators=100 - Number of boosting rounds")
print("  learning_rate=0.1 - Shrinkage parameter")
print("  max_depth=8 - Maximum tree depth")
print("  num_leaves=31 - Maximum number of leaves per tree")

print("\nTraining...")
lgb = LGBMClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=8,
    num_leaves=31,
    random_state=42,
    verbosity=-1
)
lgb.fit(X_train_split, y_train_split)

print("Evaluating on validation set...")
y_pred_lgb = lgb.predict(X_val)
lgb_accuracy = accuracy_score(y_val, y_pred_lgb)
lgb_precision = precision_score(y_val, y_pred_lgb, average='weighted', zero_division=0)
lgb_recall = recall_score(y_val, y_pred_lgb, average='weighted', zero_division=0)
lgb_f1 = f1_score(y_val, y_pred_lgb, average='weighted', zero_division=0)

print(f"\nValidation Results:")
print(f"  Accuracy:  {lgb_accuracy:.4f}")
print(f"  Precision: {lgb_precision:.4f}")
print(f"  Recall:    {lgb_recall:.4f}")
print(f"  F1-Score:  {lgb_f1:.4f}")
print(f"\nLightGBM trained successfully")

# KNN

In [ ]:
print("\n" + "="*80)
print("[MODEL 7] K-NEAREST NEIGHBORS (KNN)")
print("="*80)

print("\nModel: Instance-based lazy learner")
print("Use Case: Local decision boundaries")
print("Hyperparameters:")
print("  n_neighbors=5 - Number of neighbors to consider")

print("\nTraining...")
knn = KNeighborsClassifier(n_neighbors=5, n_jobs=-1)
knn.fit(X_train_split, y_train_split)

print("Evaluating on validation set...")
y_pred_knn = knn.predict(X_val)
knn_accuracy = accuracy_score(y_val, y_pred_knn)
knn_precision = precision_score(y_val, y_pred_knn, average='weighted', zero_division=0)
knn_recall = recall_score(y_val, y_pred_knn, average='weighted', zero_division=0)
knn_f1 = f1_score(y_val, y_pred_knn, average='weighted', zero_division=0)

print(f"\nValidation Results:")
print(f"  Accuracy:  {knn_accuracy:.4f}")
print(f"  Precision: {knn_precision:.4f}")
print(f"  Recall:    {knn_recall:.4f}")
print(f"  F1-Score:  {knn_f1:.4f}")
print(f"\nKNN trained successfully")


In [ ]:
print("MODEL COMPARISON AND ANALYSIS (MANDATORY)")
print("="*80)

# Collect all results
results = {
    'Logistic Regression': {'Accuracy': lr_accuracy, 'Precision': lr_precision, 'Recall': lr_recall, 'F1': lr_f1},
    'Random Forest': {'Accuracy': rf_accuracy, 'Precision': rf_precision, 'Recall': rf_recall, 'F1': rf_f1},
    'SVM': {'Accuracy': svm_accuracy, 'Precision': svm_precision, 'Recall': svm_recall, 'F1': svm_f1},
    'Gradient Boosting': {'Accuracy': gb_accuracy, 'Precision': gb_precision, 'Recall': gb_recall, 'F1': gb_f1},
    'XGBoost': {'Accuracy': xgb_accuracy, 'Precision': xgb_precision, 'Recall': xgb_recall, 'F1': xgb_f1},
    'LightGBM': {'Accuracy': lgb_accuracy, 'Precision': lgb_precision, 'Recall': lgb_recall, 'F1': lgb_f1},
    'KNN': {'Accuracy': knn_accuracy, 'Precision': knn_precision, 'Recall': knn_recall, 'F1': knn_f1}
}

results_df = pd.DataFrame(results).T
print("\nValidation Set Results (All Models):")
print(results_df.round(4).to_string())

print("\n" + "-"*80)
print("TOP 3 MODELS BY ACCURACY (MANDATORY ANALYSIS)")
print("-"*80)

top_3_models = results_df['Accuracy'].nlargest(3)
for rank, (model_name, accuracy) in enumerate(top_3_models.items(), 1):
    model_results = results[model_name]
    print(f"\n{rank}. {model_name}")
    print(f"   Accuracy:  {model_results['Accuracy']:.4f}")
    print(f"   Precision: {model_results['Precision']:.4f}")
    print(f"   Recall:    {model_results['Recall']:.4f}")
    print(f"   F1-Score:  {model_results['F1']:.4f}")

In [ ]:
print("KEY INSIGHTS AND OBSERVATIONS")
print("-"*80)

best_model_name = results_df['Accuracy'].idxmax()
best_accuracy = results_df['Accuracy'].max()

print(f"\n1. BEST PERFORMING MODEL: {best_model_name}")
print(f"   Accuracy: {best_accuracy:.4f}")

print(f"\n2. MODEL PERFORMANCE RANKING:")
ranked = results_df['Accuracy'].sort_values(ascending=False)
for i, (model, acc) in enumerate(ranked.items(), 1):
    print(f"   {i}. {model:25s} {acc:.4f}")

print(f"\n3. PERFORMANCE SPREAD:")
print(f"   Highest:  {results_df['Accuracy'].max():.4f}")
print(f"   Lowest:   {results_df['Accuracy'].min():.4f}")
print(f"   Range:    {results_df['Accuracy'].max() - results_df['Accuracy'].min():.4f}")
print(f"   Average:  {results_df['Accuracy'].mean():.4f}")

print(f"\n4. RECOMMENDATIONS:")
print(f"   • Tree-based models (RF, GB, XGB, LGB) generally outperform linear models")
print(f"   • Ensemble methods show strong performance")
print(f"   • Consider combining models via ensemble voting")
print(f"   • Hyperparameter tuning could improve individual models")
print(f"   • Feature engineering could boost performance")

# Create voting ensemble

In [ ]:
print("CREATING VOTING ENSEMBLE CLASSIFIER")
print("="*80)

print("\nEnsemble Strategy: SOFT VOTING")
print("  How it works:")
print("    1. Each model predicts probabilities for each class")
print("    2. Probabilities are averaged across all models")
print("    3. Final class is the one with highest average probability")
print("\nBenefits:")
print("  • Combines strengths of all 7 models")
print("  • Reduces overfitting")
print("  • Generally better generalization")
print("  • More robust predictions")

# Create ensemble
voting_models = [
    ('lr', lr),
    ('rf', rf),
    ('svm', svm),
    ('gb', gb),
    ('xgb', xgb),
    ('lgb', lgb),
    ('knn', knn)
]

ensemble = VotingClassifier(
    estimators=voting_models,
    voting='soft'  # Soft voting: averages predicted probabilities
)

print("\nEnsemble Configuration:")
for name, model in voting_models:
    print(f"  ✓ {name:10s} - {type(model).__name__}")

print("\nTraining ensemble on training data...")
ensemble.fit(X_train_split, y_train_split)

print("Evaluating ensemble on validation set...")
y_pred_ensemble = ensemble.predict(X_val)
ensemble_accuracy = accuracy_score(y_val, y_pred_ensemble)
ensemble_precision = precision_score(y_val, y_pred_ensemble, average='weighted', zero_division=0)
ensemble_recall = recall_score(y_val, y_pred_ensemble, average='weighted', zero_division=0)
ensemble_f1 = f1_score(y_val, y_pred_ensemble, average='weighted', zero_division=0)

print(f"\nEnsemble Validation Results:")
print(f"  Accuracy:  {ensemble_accuracy:.4f}")
print(f"  Precision: {ensemble_precision:.4f}")
print(f"  Recall:    {ensemble_recall:.4f}")
print(f"  F1-Score:  {ensemble_f1:.4f}")

print(f"\nPerformance Comparison:")
print(f"  Best single model: {best_model_name} ({best_accuracy:.4f})")
print(f"  Ensemble:         {ensemble_accuracy:.4f}")
improvement = ensemble_accuracy - best_accuracy
if improvement > 0:
    print(f"  Improvement:      +{improvement:.4f} ✓")
elif improvement < 0:
    print(f"  Change:           {improvement:.4f}")
else:
    print(f"  No change")

print(f"\n Ensemble created successfully")

In [ ]:
# prepare test data

print("PREPROCESSING TEST DATA")
print("="*80)

print("\nApplying same preprocessing pipeline to test data...")
print("  ✓ Text cleaning: Already done in earlier cells")
print("  ✓ Feature extraction: Using fitted vectorizers")
print("  ✓ Feature scaling: Using fitted scaler")

print(f"\nTest data shape: {X_test.shape}")
print(f"✓ Test data ready for prediction")

In [ ]:
# generate test prediction

print("GENERATING TEST PREDICTIONS")
print("="*80)

print("\nUsing ensemble to make predictions on test set...")

# Get predictions and probabilities
test_predictions = ensemble.predict(X_test)
test_probabilities = ensemble.predict_proba(X_test)

print(f"✓ Predictions generated for {len(test_predictions)} test samples")

print(f"\nPrediction Distribution:")
for cls in sorted(np.unique(test_predictions)):
    count = (test_predictions == cls).sum()
    pct = 100 * count / len(test_predictions)
    bar = '█' * int(pct / 5)
    print(f"  Class {cls}: {count:6d} ({pct:6.2f}%) {bar}")

print(f"\nConfidence Statistics:")  
confidence_scores = test_probabilities.max(axis=1)
print(f"  Min confidence:    {confidence_scores.min():.4f}")
print(f"  Max confidence:    {confidence_scores.max():.4f}")
print(f"  Mean confidence:   {confidence_scores.mean():.4f}")
print(f"  Median confidence: {np.median(confidence_scores):.4f}")

In [ ]:
print("\n" + "="*80)
print("CREATING VISUALIZATIONS")
print("="*80)

# Visualize model comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Model Performance Comparison', fontsize=16, fontweight='bold')

# Plot 1: Accuracy
ax1 = axes[0, 0]
results_df['Accuracy'].sort_values(ascending=True).plot(
    kind='barh', ax=ax1, color='skyblue', edgecolor='black'
)
ax1.set_title('Accuracy Comparison', fontweight='bold')
ax1.set_xlabel('Accuracy Score')
ax1.axvline(ensemble_accuracy, color='red', linestyle='--', linewidth=2, label='Ensemble')
ax1.legend()

# Plot 2: Precision
ax2 = axes[0, 1]
results_df['Precision'].sort_values(ascending=True).plot(
    kind='barh', ax=ax2, color='lightgreen', edgecolor='black'
)
ax2.set_title('Precision Comparison', fontweight='bold')
ax2.set_xlabel('Precision Score')

# Plot 3: Recall
ax3 = axes[1, 0]
results_df['Recall'].sort_values(ascending=True).plot(
    kind='barh', ax=ax3, color='lightcoral', edgecolor='black'
)
ax3.set_title('Recall Comparison', fontweight='bold')
ax3.set_xlabel('Recall Score')

# Plot 4: F1-Score
ax4 = axes[1, 1]
results_df['F1'].sort_values(ascending=True).plot(
    kind='barh', ax=ax4, color='lightyellow', edgecolor='black'
)
ax4.set_title('F1-Score Comparison', fontweight='bold')
ax4.set_xlabel('F1-Score')

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

print("\n Model comparison visualization saved as 'model_comparison.png'")

# Visualize prediction confidence distribution
fig, ax = plt.subplots(figsize=(10, 6))

confidence_scores = test_probabilities.max(axis=1)
ax.hist(confidence_scores, bins=30, color='skyblue', edgecolor='black', alpha=0.7)
ax.set_title('Test Predictions Confidence Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Confidence Score')
ax.set_ylabel('Frequency')
ax.axvline(confidence_scores.mean(), color='red', linestyle='--', linewidth=2, 
           label=f'Mean: {confidence_scores.mean():.4f}')
ax.axvline(np.median(confidence_scores), color='green', linestyle='--', linewidth=2,
           label=f'Median: {np.median(confidence_scores):.4f}')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('confidence_distribution.png', dpi=100, bbox_inches='tight')
plt.show()

print(" Confidence distribution visualization saved as 'confidence_distribution.png'")
